# Experiment 7 - Autoencoders, CAEs, Denoising AEs and VAEs

Source of truth is `experiment_7.py`; this notebook mirrors it.

In [ ]:
"""Experiment 7: Autoencoders, convolutional autoencoders, denoising and VAEs on MNIST."""
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json
import time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from skimage.metrics import structural_similarity

ROOT = Path(__file__).resolve().parent
SEED = 42
N_TRAIN = 10000
N_TEST = 2000
VAL_FRACTION = 0.1
LATENT_FC = 16
LATENT_CAE = 64
LATENT_VAE = 2
BATCH = 128
EPOCHS_AE = 20
EPOCHS_VAE = 20
QUICK = os.environ.get('EXP7_QUICK') == '1'
if QUICK:
    N_TRAIN, N_TEST, EPOCHS_AE, EPOCHS_VAE = 1000, 500, 2, 2

np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
NOISE_RNG = np.random.default_rng(SEED)


def load_mnist():
    (x_train_full, _), (x_test_full, y_test) = tf.keras.datasets.mnist.load_data()
    x_train = (x_train_full[:N_TRAIN].astype('float32') / 255.0)[..., None]
    x_test = (x_test_full[:N_TEST].astype('float32') / 255.0)[..., None]
    n_val = int(N_TRAIN * VAL_FRACTION)
    x_val = x_train[:n_val]
    x_fit = x_train[n_val:]
    return x_fit, x_val, x_test, y_test[:N_TEST]


def add_gaussian(x, sigma):
    noise = NOISE_RNG.normal(0.0, sigma, x.shape).astype('float32')
    return np.clip(x + noise, 0.0, 1.0)


def add_salt_pepper(x, p):
    corrupted = x.copy()
    draw = NOISE_RNG.random(x.shape)
    corrupted[draw < p / 2.0] = 0.0
    corrupted[(draw >= p / 2.0) & (draw < p)] = 1.0
    return corrupted


def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(ROOT / name, dpi=130, bbox_inches='tight')
    plt.close(fig)


def recon_metrics(x_true, x_hat):
    mse = float(np.mean((x_true - x_hat) ** 2))
    mae = float(np.mean(np.abs(x_true - x_hat)))
    ssim = float(np.mean([
        structural_similarity(x_true[i, :, :, 0], x_hat[i, :, :, 0], data_range=1.0)
        for i in range(len(x_true))
    ]))
    return mse, mae, ssim


def show_grid(images, titles, nrow, ncol, suptitle, fname, figsize=None, cmap='gray'):
    fig, axes = plt.subplots(nrow, ncol, figsize=figsize or (1.4 * ncol, 1.6 * nrow))
    axes = np.atleast_2d(axes)
    for ax in axes.flat:
        ax.axis('off')
    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img.reshape(28, 28), cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title, fontsize=7)
    fig.suptitle(suptitle, fontsize=11)
    save_fig(fig, fname)


def build_fc_ae(latent=LATENT_FC):
    inp = layers.Input(shape=(784,), name='fc_input')
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    z = layers.Dense(latent, activation='relu', name='latent')(x)
    x = layers.Dense(32, activation='relu')(z)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(784, activation='sigmoid')(x)
    model = models.Model(inp, out, name='fc_autoencoder')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    return model


def build_cae():
    inp = layers.Input(shape=(28, 28, 1), name='cae_input')
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2, padding='same')(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)
    z = layers.Conv2D(LATENT_CAE, 3, activation='relu', padding='same', name='latent_map')(x)
    x = layers.UpSampling2D(2)(z)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.UpSampling2D(2)(x)
    out = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)
    model = models.Model(inp, out, name='conv_autoencoder')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    return model


class Sampling(layers.Layer):
    def call(self, inputs):
        mu, logvar = inputs
        eps = tf.random.normal(shape=tf.shape(mu))
        return mu + tf.exp(0.5 * logvar) * eps


class KLDivergence(layers.Layer):
    def call(self, inputs):
        mu, logvar = inputs
        self.add_loss(-0.5 * tf.reduce_mean(tf.reduce_sum(1 + logvar - tf.square(mu) - tf.exp(logvar), axis=-1)))
        return mu


def build_vae(latent=LATENT_VAE):
    inp = layers.Input(shape=(28, 28, 1), name='vae_input')
    x = layers.Conv2D(32, 3, strides=2, activation='relu', padding='same')(inp)
    x = layers.Conv2D(64, 3, strides=2, activation='relu', padding='same')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(16, activation='relu')(x)
    mu = layers.Dense(latent, name='mu')(x)
    logvar = layers.Dense(latent, name='logvar')(x)
    z = Sampling()([mu, logvar])
    KLDivergence()([mu, logvar])
    # Decoder layers are created once and reused, so the VAE and the standalone
    # decoder share the same weights.
    dec_dense = layers.Dense(7 * 7 * 64, activation='relu')
    dec_reshape = layers.Reshape((7, 7, 64))
    dec_deconv1 = layers.Conv2DTranspose(64, 3, strides=2, activation='relu', padding='same')
    dec_deconv2 = layers.Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same')
    dec_out = layers.Conv2D(1, 3, activation='sigmoid', padding='same')

    def decode(t):
        t = dec_dense(t)
        t = dec_reshape(t)
        t = dec_deconv1(t)
        t = dec_deconv2(t)
        return dec_out(t)

    vae = models.Model(inp, decode(z), name='vae')
    vae.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
    encoder = models.Model(inp, [mu, logvar], name='vae_encoder')
    decoder_in = layers.Input(shape=(latent,), name='decoder_input')
    decoder = models.Model(decoder_in, decode(decoder_in), name='vae_decoder')
    return vae, encoder, decoder


def train(model, x_in, x_target, x_val, x_val_target, epochs, tag):
    start = time.perf_counter()
    history = model.fit(x_in, x_target, validation_data=(x_val, x_val_target),
                        epochs=epochs, batch_size=BATCH, verbose=0)
    elapsed = time.perf_counter() - start
    print(f"  {tag}: {elapsed:.1f}s  final loss {history.history['loss'][-1]:.4f}  val {history.history['val_loss'][-1]:.4f}")
    return history, elapsed

In [ ]:
results = {}
(x_fit, x_val, x_test, y_test) = load_mnist()
y_test = np.asarray(y_test)
print(f"train {x_fit.shape}  val {x_val.shape}  test {x_test.shape}")

results.update({
    "seed": SEED, "n_train": int(len(x_fit) + len(x_val)), "n_val": int(len(x_val)),
    "n_test": int(len(x_test)), "batch": BATCH, "epochs_ae": EPOCHS_AE, "epochs_vae": EPOCHS_VAE,
    "quick": QUICK,
})

x_fit_flat, x_val_flat, x_test_flat = x_fit.reshape(-1, 784), x_val.reshape(-1, 784), x_test.reshape(-1, 784)

## Dataset and samples

Load MNIST, inspect the test digit distribution.

In [ ]:
# ---------------- sample images and class distribution ----------------
show_grid(list(x_fit[:10]), [""] * 10, 2, 5, "Sample MNIST images (28x28)", "exp7_plot_01_sample_images.png")
counts = np.bincount(y_test, minlength=10)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(10), counts, color='steelblue', edgecolor='black')
ax.set_xticks(range(10))
ax.set_xlabel("digit")
ax.set_ylabel("count")
ax.set_title(f"Digit distribution of the {len(x_test)}-image test subset")
save_fig(fig, "exp7_plot_02_class_distribution.png")

## Fully connected autoencoder

784-128-32-16 latent, sigmoid output, BCE loss, Adam 1e-3.

In [ ]:
# ---------------- fully connected autoencoder ----------------
print("Fully connected autoencoder")
fcae = build_fc_ae()
hist_fcae, time_fcae = train(fcae, x_fit_flat, x_fit_flat, x_val_flat, x_val_flat, EPOCHS_AE, "FC-AE")
x_test_fc = fcae.predict(x_test_flat, batch_size=256, verbose=0).reshape(-1, 28, 28, 1)
mse_fc, mae_fc, ssim_fc = recon_metrics(x_test, x_test_fc)
params_fc = int(fcae.count_params())
results["fc_ae"] = {"mse": mse_fc, "mae": mae_fc, "ssim": ssim_fc, "params": params_fc,
                    "time_s": time_fcae, "loss_curve": hist_fcae.history['loss'],
                    "val_loss_curve": hist_fcae.history['val_loss']}

idx = np.arange(10)
images, titles = [], []
for i in idx:
    images += [x_test[i], x_test_fc[i]]
    titles += [f"orig {y_test[i]}", f"recon {y_test[i]}"]
show_grid(images, titles, 4, 5, "Plot 1: FC-AE original vs reconstructed (test)", "exp7_plot_03_fcae_recon.png")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hist_fcae.history['loss'], label='training loss', color='steelblue')
ax.plot(hist_fcae.history['val_loss'], label='validation loss', color='darkorange')
ax.set_title("Plot 2: FC-AE reconstruction loss")
ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, "exp7_plot_04_fcae_loss.png")

## Convolutional autoencoder

Same protocol as the FC-AE so the two are comparable.

In [ ]:
# ---------------- convolutional autoencoder ----------------
print("Convolutional autoencoder")
cae = build_cae()
hist_cae, time_cae = train(cae, x_fit, x_fit, x_val, x_val, EPOCHS_AE, "CAE")
x_test_cae = cae.predict(x_test, batch_size=256, verbose=0)
mse_cae, mae_cae, ssim_cae = recon_metrics(x_test, x_test_cae)
params_cae = int(cae.count_params())
results["cae"] = {"mse": mse_cae, "mae": mae_cae, "ssim": ssim_cae, "params": params_cae,
                  "time_s": time_cae, "loss_curve": hist_cae.history['loss'],
                  "val_loss_curve": hist_cae.history['val_loss']}

images, titles = [], []
for i in idx:
    images += [x_test[i], x_test_fc[i], x_test_cae[i]]
    titles += [f"orig {y_test[i]}", f"FC-AE {y_test[i]}", f"CAE {y_test[i]}"]
show_grid(images, titles, 5, 6, "Plot 3: Original / FC-AE / CAE reconstruction", "exp7_plot_05_fc_vs_cae.png")

## Denoising convolutional autoencoder

Train on Gaussian-noisy inputs with clean targets.

In [ ]:
# ---------------- denoising autoencoder ----------------
print("Denoising convolutional autoencoder")
sigma_train = 0.2
x_fit_noisy = add_gaussian(x_fit, sigma_train)
x_val_noisy = add_gaussian(x_val, sigma_train)
dae = build_cae()
hist_dae, time_dae = train(dae, x_fit_noisy, x_fit, x_val_noisy, x_val, EPOCHS_AE, "DAE(gauss)")

dae_metrics = {}
for sigma in (0.1, 0.2, 0.3):
    noisy = add_gaussian(x_test, sigma)
    denoised = dae.predict(noisy, batch_size=256, verbose=0)
    m, a, s = recon_metrics(x_test, denoised)
    noise_m, _, noise_s = recon_metrics(x_test, noisy)
    dae_metrics[f"gauss_{sigma}"] = {"mse": m, "mae": a, "ssim": s,
                                     "noisy_mse": noise_m, "noisy_ssim": noise_s}
    print(f"  gaussian sigma {sigma}: mse {m:.5f} ssim {s:.4f} (noisy ssim {noise_s:.4f})")

sp_metrics = {}
for p in (0.05, 0.10, 0.20):
    noisy = add_salt_pepper(x_test, p)
    denoised = dae.predict(noisy, batch_size=256, verbose=0)
    m, a, s = recon_metrics(x_test, denoised)
    sp_metrics[f"sp_{p}"] = {"mse": m, "mae": a, "ssim": s}
    print(f"  salt-pepper p {p}: mse {m:.5f} ssim {s:.4f}")

results["dae"] = {"params": int(dae.count_params()), "time_s": time_dae,
                  "train_sigma": sigma_train, "gaussian": dae_metrics, "salt_pepper": sp_metrics,
                  "loss_curve": hist_dae.history['loss'], "val_loss_curve": hist_dae.history['val_loss']}

# Plot 4: clean / noisy / denoised grid at sigma 0.2
noisy02 = add_gaussian(x_test, 0.2)
denoised02 = dae.predict(noisy02, batch_size=256, verbose=0)
images, titles = [], []
for i in idx:
    images += [x_test[i], noisy02[i], denoised02[i]]
    titles += [f"clean {y_test[i]}", "noisy", "denoised"]
show_grid(images, titles, 5, 6, "Plot 4: clean / Gaussian-noisy / denoised", "exp7_plot_06_clean_noisy_denoised.png")

# Plot 5: noise level vs metrics, three panels
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sigmas = [0.1, 0.2, 0.3]
axes[0].plot(sigmas, [dae_metrics[f"gauss_{s}"]["mse"] for s in sigmas], 'o-', color='steelblue', label='denoised')
axes[0].plot(sigmas, [dae_metrics[f"gauss_{s}"]["noisy_mse"] for s in sigmas], 's--', color='crimson', label='noisy input')
axes[0].set_xlabel("Gaussian sigma"); axes[0].set_ylabel("MSE"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(sigmas, [dae_metrics[f"gauss_{s}"]["mae"] for s in sigmas], 'o-', color='seagreen')
axes[1].set_xlabel("Gaussian sigma"); axes[1].set_ylabel("MAE"); axes[1].grid(alpha=0.3)
axes[2].plot(sigmas, [dae_metrics[f"gauss_{s}"]["ssim"] for s in sigmas], 'o-', color='darkorange', label='denoised')
axes[2].plot(sigmas, [dae_metrics[f"gauss_{s}"]["noisy_ssim"] for s in sigmas], 's--', color='crimson', label='noisy input')
axes[2].set_xlabel("Gaussian sigma"); axes[2].set_ylabel("SSIM"); axes[2].legend(); axes[2].grid(alpha=0.3)
fig.suptitle("Plot 5: Noise level vs reconstruction quality")
save_fig(fig, "exp7_plot_07_noise_level_metrics.png")

# Additional: salt-and-pepper vs gaussian SSIM comparison
fig, ax = plt.subplots(figsize=(8, 4.5))
ps = [0.05, 0.10, 0.20]
ax.plot(sigmas, [dae_metrics[f"gauss_{s}"]["ssim"] for s in sigmas], 'o-', color='steelblue', label='Gaussian (matched family)')
ax.plot([p * 2 for p in ps], [sp_metrics[f"sp_{p}"]["ssim"] for p in ps], 's-', color='crimson', label='Salt-pepper (unseen family)')
ax.set_xlabel("approximate corruption strength")
ax.set_ylabel("SSIM after denoising")
ax.set_title("Denoising cross-noise-type comparison")
ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, "exp7_plot_15_dae_noise_compare.png")

## Variational autoencoder

Latent dimension 2, reparameterization trick, BCE plus KL.

In [ ]:
# ---------------- variational autoencoder ----------------
print("Variational autoencoder")
vae, encoder, decoder = build_vae()
hist_vae, time_vae = train(vae, x_fit, x_fit, x_val, x_val, EPOCHS_VAE, "VAE")
x_test_vae = vae.predict(x_test, batch_size=256, verbose=0)
mse_vae, mae_vae, ssim_vae = recon_metrics(x_test, x_test_vae)
mu_test, logvar_test = encoder.predict(x_test, batch_size=256, verbose=0)

def bce_sum(a, b):
    b = np.clip(b, 1e-7, 1 - 1e-7)
    return float(np.mean(np.sum(-(a * np.log(b) + (1 - a) * np.log(1 - b)), axis=(1, 2, 3))))

recon_loss_vae = bce_sum(x_test, x_test_vae)
kl_loss_vae = float(np.mean(-0.5 * np.sum(1 + logvar_test - mu_test ** 2 - np.exp(logvar_test), axis=1)))
params_vae = int(vae.count_params())
results["vae"] = {"mse": mse_vae, "mae": mae_vae, "ssim": ssim_vae, "params": params_vae,
                  "time_s": time_vae, "recon_loss": recon_loss_vae, "kl_loss": kl_loss_vae,
                  "total_loss": recon_loss_vae + kl_loss_vae,
                  "loss_curve": hist_vae.history['loss'], "val_loss_curve": hist_vae.history['val_loss']}
print(f"  VAE recon {recon_loss_vae:.2f}  KL {kl_loss_vae:.2f}  mse {mse_vae:.5f}  ssim {ssim_vae:.4f}")

# Plot: VAE latent space
fig, ax = plt.subplots(figsize=(7.5, 6.5))
sc = ax.scatter(mu_test[:, 0], mu_test[:, 1], c=y_test, cmap='tab10', s=8, alpha=0.7)
fig.colorbar(sc, ax=ax, ticks=range(10), label='digit')
ax.set_xlabel("$z_1$"); ax.set_ylabel("$z_2$")
ax.set_title("Plot 6: VAE latent space of test images")
ax.grid(alpha=0.2)
save_fig(fig, "exp7_plot_08_vae_latent_space.png")

# Plot: randomly generated images
gen_z = np.random.default_rng(SEED).normal(0, 1, (25, LATENT_VAE)).astype('float32')
samples = decoder.predict(gen_z, verbose=0)
show_grid(list(samples), [""] * 25, 5, 5, "Plot 7: VAE generated digits from $z\\sim N(0,I)$",
          "exp7_plot_09_vae_generated.png", figsize=(6, 6.5))

# Plot: latent interpolation
z_a = mu_test[np.where(y_test == 0)[0][0]]
z_b = mu_test[np.where(y_test == 1)[0][0]]
alphas = np.linspace(0, 1, 11)
interp_z = np.array([(1 - a) * z_a + a * z_b for a in alphas]).astype('float32')
interp_imgs = decoder.predict(interp_z, verbose=0)
fig, axes = plt.subplots(1, 11, figsize=(15, 2))
for ax, img, a in zip(axes, interp_imgs, alphas):
    ax.imshow(img.reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f"{a:.1f}", fontsize=7)
    ax.axis('off')
fig.suptitle("Plot 8: latent interpolation from a 0 to a 1")
save_fig(fig, "exp7_plot_10_vae_interpolation.png")

# Plot: VAE training/validation loss
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hist_vae.history['loss'], label='training loss (recon + KL)', color='steelblue')
ax.plot(hist_vae.history['val_loss'], label='validation loss', color='darkorange')
ax.set_title("Plot 9: VAE training and validation loss")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, "exp7_plot_11_vae_loss.png")

# Plot 10: reconstruction-error distribution (FC-AE and CAE)
err_fc = np.mean((x_test - x_test_fc) ** 2, axis=(1, 2, 3))
err_cae = np.mean((x_test - x_test_cae) ** 2, axis=(1, 2, 3))
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(err_fc, bins=50, alpha=0.6, label='FC-AE', color='steelblue')
ax.hist(err_cae, bins=50, alpha=0.6, label='CAE', color='darkorange')
ax.set_title("Plot 10: Per-image reconstruction-error distribution (test)")
ax.set_xlabel("MSE per image"); ax.set_ylabel("count"); ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, "exp7_plot_12_error_distribution.png")

# high-error images from CAE
top5 = np.argsort(err_cae)[::-1][:5]
images, titles = [], []
for i in top5:
    images += [x_test[i], x_test_cae[i]]
    titles += [f"orig {y_test[i]}", f"recon {y_test[i]}"]
show_grid(images, titles, 2, 5, "High reconstruction-error test images (CAE)", "exp7_plot_13_high_error.png")
results["high_error_idx"] = top5.tolist()
results["high_error_labels"] = [int(y_test[i]) for i in top5]

## Latent dimension study

FC-AE with dz in {2, 8, 16, 32}.

In [ ]:
# ---------------- latent dimension study on FC-AE ----------------
print("Latent dimension study")
dim_study = {}
for dz in (2, 8, 16, 32):
    m = build_fc_ae(dz)
    _, t = train(m, x_fit_flat, x_fit_flat, x_val_flat, x_val_flat, EPOCHS_AE, f"FC-AE dz={dz}")
    recon = m.predict(x_test_flat, batch_size=256, verbose=0).reshape(-1, 28, 28, 1)
    mm, _, ss = recon_metrics(x_test, recon)
    dim_study[str(dz)] = {"mse": mm, "ssim": ss, "params": int(m.count_params()), "time_s": t}
    print(f"  dz {dz}: mse {mm:.5f} ssim {ss:.4f} params {m.count_params()}")
results["latent_dim_study"] = dim_study

dims = [2, 8, 16, 32]
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(dims, [dim_study[str(d)]["mse"] for d in dims], 'o-', color='steelblue')
ax1.set_xlabel("latent dimension $d_z$"); ax1.set_ylabel("test MSE", color='steelblue')
ax1.set_title("Latent dimension vs reconstruction MSE")
ax1.grid(alpha=0.3)
save_fig(fig, "exp7_plot_14_latent_dim_study.png")

## Consolidated results

Write results JSON and metrics summary.

In [ ]:
# ---------------- consolidated tables ----------------
results["comparison"] = {
    "FC-AE": {"mse": mse_fc, "mae": mae_fc, "ssim": ssim_fc, "params": params_fc},
    "CAE": {"mse": mse_cae, "mae": mae_cae, "ssim": ssim_cae, "params": params_cae},
    "DAE": {"mse": dae_metrics["gauss_0.2"]["mse"], "mae": dae_metrics["gauss_0.2"]["mae"],
            "ssim": dae_metrics["gauss_0.2"]["ssim"], "params": int(dae.count_params())},
    "VAE": {"mse": mse_vae, "mae": mae_vae, "ssim": ssim_vae, "params": params_vae},
}

(ROOT / "experiment_7_results.json").write_text(json.dumps(results, indent=2))

lines = []
lines.append("Experiment 7 reconstruction metrics (test subset, seed 42)")
lines.append("=" * 60)
lines.append(f"{'Model':<24}{'MSE':>10}{'MAE':>10}{'SSIM':>9}{'Params':>10}{'Time(s)':>10}")
for name, r in results["comparison"].items():
    t = {"FC-AE": time_fcae, "CAE": time_cae, "DAE": time_dae, "VAE": time_vae}[name]
    lines.append(f"{name:<24}{r['mse']:>10.5f}{r['mae']:>10.5f}{r['ssim']:>9.4f}{r['params']:>10}{t:>10.1f}")
lines.append("")
lines.append(f"VAE reconstruction loss {recon_loss_vae:.3f}  KL loss {kl_loss_vae:.3f}  total {recon_loss_vae + kl_loss_vae:.3f}")
lines.append("")
lines.append("Gaussian denoising (train sigma 0.2):")
for s in (0.1, 0.2, 0.3):
    d = dae_metrics[f"gauss_{s}"]
    lines.append(f"  sigma {s}: MSE {d['mse']:.5f}  MAE {d['mae']:.5f}  SSIM {d['ssim']:.4f}")
lines.append("Salt-and-pepper denoising (train family Gaussian):")
for p in (0.05, 0.10, 0.20):
    d = sp_metrics[f"sp_{p}"]
    lines.append(f"  p {p}: MSE {d['mse']:.5f}  MAE {d['mae']:.5f}  SSIM {d['ssim']:.4f}")
lines.append("")
lines.append("Latent dimension study (FC-AE):")
for d in dims:
    r = dim_study[str(d)]
    lines.append(f"  dz {d}: MSE {r['mse']:.5f}  SSIM {r['ssim']:.4f}  params {r['params']}")
(ROOT / "experiment_7_metrics.txt").write_text("\n".join(lines) + "\n")

print("Done. Results and figures written.")